# LayaChess (fine-tuned + search) vs Stockfish ladder

Fine-tuned Laya (`datafreak/laya-chess`) with MCTS search plays Stockfish at several `UCI_Elo` levels.
Both T4s play games in parallel (one engine per GPU). Every finished game is saved immediately to `/kaggle/working/matches/`.

**Run:** GPU **T4 x2**, Internet **ON**, secret `HF_TOKEN` attached (needed for the private model). Then **Save & Run All (Commit)**.

In [ ]:
# ===== Settings =====
LEVELS = [1320, 1500, 1700, 1900, 2100]   # Stockfish UCI_Elo levels
GAMES_PER_LEVEL = 24                      # 12 openings x both colours
LAYA_NODES = 32                           # positions Laya searches per move (fixed -> same strength on any GPU)
SF_SECONDS = 0.3                          # Stockfish time per move
TIME_BUDGET_H = 5.75                      # stop starting new games after this (~7 h quota left: setup + last games + upload fit)
CHECKPOINT = "datafreak/laya-chess"
REVISION = None                           # pin a commit for exact reproducibility
MAX_PLIES = 300
UPLOAD_TO_HF = True                       # push PGNs + results to the model repo under matches/ (every 30 min + at the end)
BACKUP_EVERY_MIN = 30

In [ ]:
!pip -q install python-chess "git+https://github.com/NandhaKishorM/laya.git"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Official Stockfish build (UCI_Elo down to 1320), pinned. Stops the notebook here if it doesn't work,
# so no GPU time is spent on games that can't be played.
import os, shutil, subprocess, tarfile, urllib.request
SF_BUILDS = [  # (url, path of the binary inside the archive)
    ("https://github.com/official-stockfish/Stockfish/releases/download/sf_19/stockfish-linux-x86-64-universal.tar.gz",
     "stockfish/stockfish-linux-x86-64-universal"),
    ("https://github.com/official-stockfish/Stockfish/releases/download/sf_18/stockfish-ubuntu-x86-64-avx2.tar",
     "stockfish/stockfish-ubuntu-x86-64-avx2"),
]
for url, member in SF_BUILDS:
    try:
        archive = "/tmp/" + url.rsplit("/", 1)[1]
        urllib.request.urlretrieve(url, archive)
        with tarfile.open(archive) as t:
            t.extract(member, "/tmp/sf")
        shutil.copy(f"/tmp/sf/{member}", "/usr/local/bin/stockfish"); os.chmod("/usr/local/bin/stockfish", 0o755)
        out = subprocess.run(["stockfish"], input="uci\nquit\n", capture_output=True, text=True, timeout=30).stdout
        assert "uciok" in out and "UCI_Elo" in out
        print([l for l in out.splitlines() if l.startswith("id name") or "UCI_Elo" in l])
        break
    except Exception as e:
        print("Stockfish build failed:", url, e)
else:
    raise SystemExit("No working Stockfish -> stopping before any GPU time is used")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found")
except Exception as e:
    raise SystemExit("Attach the HF_TOKEN secret (Add-ons -> Secrets): the model repo is private") from e

# check the token before any GPU work: valid, can see the model, can write (for result backups)
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
try:
    who = api.whoami()
    api.model_info(CHECKPOINT)
except Exception as e:
    raise SystemExit(f"HF_TOKEN doesn't work ({type(e).__name__}): create a new token on huggingface.co "
                     f"and update the Kaggle secret (Add-ons -> Secrets -> HF_TOKEN -> Edit)") from e
role = (who.get("auth", {}).get("accessToken", {}) or {}).get("role", "?")
print(f"Hugging Face: logged in as {who['name']}, token role: {role}, model {CHECKPOINT} reachable")
if UPLOAD_TO_HF and role == "read":
    print("WARNING: read-only token -> games run fine but backups to Hugging Face will fail; use a Write token")

## Engine code (same files as `engine/laya_chess/` in the repo)

In [ ]:
import os
os.makedirs('/kaggle/working/engine/laya_chess', exist_ok=True)
FILES = {'encoding.py': '"""Position + move -> Laya `score` question. Must match the training notebook exactly."""\nimport chess\nimport numpy as np\n\nNAMES = {chess.PAWN: "pawn", chess.KNIGHT: "knight", chess.BISHOP: "bishop",\n         chess.ROOK: "rook", chess.QUEEN: "queen", chess.KING: "king"}\nORDER = [chess.KING, chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT, chess.PAWN]\n\nDEFAULT_N_LEVELS = 10\nDEFAULT_INS_TMPL = "{side} plays {san} ({piece} {frm}-{to}{extra}). Win chance for {side}?"\n\n\ndef piece_list(board, color):\n    out = []\n    for pt in ORDER:\n        for sq in sorted(board.pieces(pt, color)):\n            out.append(chess.piece_symbol(pt).upper() + chess.square_name(sq))\n    return " ".join(out)\n\n\ndef board_state(board):\n    return {\n        "to_move": "white" if board.turn else "black",\n        "white": piece_list(board, chess.WHITE),\n        "black": piece_list(board, chess.BLACK),\n        "castling": board.castling_xfen() if board.castling_rights else "-",\n        "en_passant": chess.square_name(board.ep_square) if board.has_legal_en_passant() else "-",\n    }\n\n\nclass MoveEncoder:\n    """Builds the token ids for "how good is `move` in `board`?" exactly as in training.\n\n    Level labels and the instruction template come from the checkpoint\'s chess_meta.json, so the\n    engine always asks the question the model was trained on.\n    """\n\n    def __init__(self, tokenizer, laya_cfg, meta=None):\n        from laya.common import build_sequence\n        meta = meta or {}\n        if meta.get("state_format", "piece lists v2") != "piece lists v2":\n            raise ValueError(f"checkpoint uses state format {meta[\'state_format\']!r}, engine supports \'piece lists v2\'")\n        self.n_levels = int(meta.get("n_levels", DEFAULT_N_LEVELS))\n        edges = np.linspace(0, 1, self.n_levels + 1)\n        self.centers = ((edges[:-1] + edges[1:]) / 2).astype(np.float32)\n        self.crit = meta.get("crit") or [f"{edges[i]*100:.0f}-{edges[i+1]*100:.0f}%" for i in range(self.n_levels)]\n        self.ins_tmpl = meta.get("ins_template") or DEFAULT_INS_TMPL\n        self._build_sequence = build_sequence\n        self.tokenizer, self.max_len, self.head_max_len = tokenizer, laya_cfg["max_len"], laya_cfg["head_max_len"]\n\n    def question(self, board, move):\n        piece = board.piece_at(move.from_square)\n        extra = ""\n        if board.is_capture(move):\n            cap = board.piece_at(move.to_square)\n            extra += f", takes {NAMES[cap.piece_type] if cap else \'pawn\'}"\n        if move.promotion:\n            extra += f", promotes to {NAMES[move.promotion]}"\n        if board.gives_check(move):\n            extra += ", check"\n        side = "white" if board.turn else "black"\n        ins = self.ins_tmpl.format(side=side, san=board.san(move), piece=NAMES[piece.piece_type],\n                                   frm=chess.square_name(move.from_square), to=chess.square_name(move.to_square),\n                                   extra=extra)\n        return {"t": "score", "ins": ins, "crit": self.crit}\n\n    def encode(self, board, move, state=None):\n        state = state if state is not None else board_state(board)\n        ids, markers = self._build_sequence(self.tokenizer, state, self.question(board, move),\n                                            self.max_len, self.head_max_len)[:2]\n        return ids, markers\n', 'model.py': '"""Load a Laya chess checkpoint and score every legal move of a position."""\nimport json\nimport os\nimport time\nfrom collections import OrderedDict\nfrom pathlib import Path\n\nimport chess\nimport numpy as np\nimport torch\n\nfrom .encoding import MoveEncoder, board_state\n\nDEFAULT_CHECKPOINT = "datafreak/laya-chess"\n# only what inference needs (skips the ~3.4 GB optimizer state and other Laya variants)\nALLOW = ["model.safetensors", "rl_agent_config.json", "chess_meta.json", "encoder/*", "tokenizer/*"]\n\n\ndef pick_device(name="auto"):\n    if name != "auto":\n        return torch.device(name)\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef position_key(board):\n    # position + side + castling + en passant; move counters don\'t change the encoding\n    return board._transposition_key()\n\n\nclass LayaChessModel:\n    """`evaluate(boards)` -> for each board: (legal moves, win chance for the side to move after each move).\n\n    `checkpoint` is a Hugging Face repo id (private repos use your `hf auth login` token) or a local folder.\n    """\n\n    def __init__(self, checkpoint=DEFAULT_CHECKPOINT, revision=None, device="auto", dtype=None,\n                 batch_size=96, cache_size=200_000, verbose=True):\n        from huggingface_hub import snapshot_download\n        from laya.agent import _fix_tokenizer_config\n        from laya.common import QTYPES, build_model\n        from safetensors.torch import load_file\n        from transformers import AutoTokenizer\n\n        t0 = time.time()\n        path = Path(checkpoint) if os.path.isdir(checkpoint) else Path(\n            snapshot_download(checkpoint, revision=revision, allow_patterns=ALLOW))\n        _fix_tokenizer_config(str(path))\n        laya_cfg = json.loads((path / "rl_agent_config.json").read_text())\n        meta_file = path / "chess_meta.json"\n        self.meta = json.loads(meta_file.read_text()) if meta_file.exists() else {}\n        self.checkpoint = checkpoint\n\n        self.tokenizer = AutoTokenizer.from_pretrained(path / "tokenizer")\n        self.encoder = MoveEncoder(self.tokenizer, laya_cfg, self.meta)\n        model = build_model(laya_cfg, encoder_dir=str(path / "encoder"), pretrained=False)\n        for m in model.modules():  # no torch.compile inside ModernBERT (slow first call, breaks on MPS)\n            if hasattr(m, "config") and hasattr(m.config, "reference_compile"):\n                m.config.reference_compile = False\n        model.load_state_dict({k: v.float() for k, v in load_file(str(path / "model.safetensors")).items()}, strict=True)\n\n        self.device = pick_device(device)\n        self.dtype = dtype or (torch.float16 if self.device.type in ("cuda", "mps") else torch.float32)\n        self.model = model.to(self.device, self.dtype).eval()\n        self.qtype = QTYPES["score"]\n        self.pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0\n        self.centers = torch.tensor(self.encoder.centers, device=self.device)\n        self.batch_size = batch_size\n        self.cache, self.cache_size = OrderedDict(), cache_size\n        self.n_positions = self.n_sequences = 0\n        if verbose:\n            tag = self.meta.get("tag", "base Laya (untrained for chess)")\n            print(f"[laya-chess] {checkpoint} ({tag}, step {self.meta.get(\'step\', 0)}, "\n                  f"{self.meta.get(\'consumed\', 0):,} examples) on {self.device}/{str(self.dtype).split(\'.\')[-1]} "\n                  f"in {time.time() - t0:.1f}s", flush=True)\n\n    @torch.no_grad()\n    def _forward(self, items):\n        L = max(len(ids) for ids, _ in items)\n        M = max(len(mk) for _, mk in items)\n        B = len(items)\n        ids = torch.full((B, L), self.pad_id, dtype=torch.long)\n        att = torch.zeros((B, L), dtype=torch.long)\n        pos = torch.zeros((B, M), dtype=torch.long)\n        mask = torch.zeros((B, M), dtype=torch.bool)\n        for j, (i, mk) in enumerate(items):\n            ids[j, :len(i)] = torch.tensor(i)\n            att[j, :len(i)] = 1\n            pos[j, :len(mk)] = torch.tensor(mk)\n            mask[j, :len(mk)] = True\n        qt = torch.full((B,), self.qtype, dtype=torch.long)\n        ids, att, pos, mask, qt = (t.to(self.device) for t in (ids, att, pos, mask, qt))\n        logits, _ = self.model(ids, att, pos, mask, qt)\n        p = torch.softmax(logits.float().masked_fill(~mask, -1e4), -1)[:, :len(self.centers)]\n        return (p * self.centers).sum(-1).cpu().numpy()\n\n    def evaluate(self, boards):\n        """Batch-score all legal moves of several positions in as few forward passes as possible."""\n        out, todo = [None] * len(boards), []\n        for bi, board in enumerate(boards):\n            key = position_key(board)\n            if key in self.cache:\n                self.cache.move_to_end(key)\n                out[bi] = self.cache[key]\n            else:\n                todo.append((bi, key, board, list(board.legal_moves)))\n        items, owners = [], []\n        for bi, key, board, moves in todo:\n            state = board_state(board)\n            for mv in moves:\n                items.append(self.encoder.encode(board, mv, state))\n                owners.append(bi)\n        values = np.concatenate([self._forward(items[k:k + self.batch_size])\n                                 for k in range(0, len(items), self.batch_size)]) if items else np.zeros(0)\n        k = 0\n        for bi, key, board, moves in todo:\n            q = values[k:k + len(moves)].astype(np.float32); k += len(moves)\n            out[bi] = (moves, q)\n            self.cache[key] = out[bi]\n            if len(self.cache) > self.cache_size:\n                self.cache.popitem(last=False)\n        self.n_positions += len(todo); self.n_sequences += len(items)\n        return out\n\n    def score_moves(self, board):\n        """Moves sorted best-first with the model\'s win chance for the side to move (no search)."""\n        moves, q = self.evaluate([board])[0]\n        order = np.argsort(-q)\n        return [(moves[i], float(q[i])) for i in order]\n', 'search.py': '"""Monte Carlo tree search (PUCT, AlphaZero/Leela style) on top of Laya\'s move scores.\n\nLaya gives, for every legal move a of a position s, the win chance Q(s, a) for the side to move.\nOne evaluation of s therefore yields:\n  * priors  P(a) = softmax(Q(s, a) / prior_temp)   -> which moves to explore first\n  * a value V(s) = max_a Q(s, a)                    -> how good s is for the side to move\n  * per-move initial values Q(s, a)                 -> used as one "virtual visit" per edge\nCheckmate / stalemate / draws are scored exactly from the rules, never by the network.\n"""\nimport math\nimport threading\nimport time\nfrom dataclasses import dataclass, field\n\nimport chess\nimport numpy as np\n\nWIN, DRAW, LOSS = 1.0, 0.5, 0.0\n\n\ndef terminal_value(board):\n    """Exact value for the side to move if the game is over, else None."""\n    outcome = board.outcome(claim_draw=True)\n    if outcome is None:\n        return None\n    if outcome.winner is None:\n        return DRAW\n    return WIN if outcome.winner == board.turn else LOSS\n\n\ndef winprob_to_cp(p):\n    """Inverse of the Lichess win% formula ChessBench used (win% = 1 / (1 + exp(-0.00368208 * cp)))."""\n    p = min(max(p, 1e-4), 1 - 1e-4)\n    return int(round(math.log(p / (1 - p)) / 0.00368208))\n\n\n@dataclass\nclass Node:\n    moves: list\n    q_net: np.ndarray            # network win chance per move (side to move)\n    prior: np.ndarray\n    N: np.ndarray                # visits per move\n    W: np.ndarray                # summed values per move (side to move\'s view)\n    exact: np.ndarray            # NaN, or exact value of the move (mate / stalemate / draw)\n    children: dict = field(default_factory=dict)\n    visits: int = 0\n\n    def q(self):\n        """Mean value per move: the network estimate counts as one visit, exact results override."""\n        q = (self.q_net + self.W) / (1.0 + self.N)\n        return np.where(np.isnan(self.exact), q, self.exact)\n\n\n@dataclass\nclass SearchResult:\n    move: chess.Move\n    value: float                 # expected win chance for the side to move\n    nodes: int                   # positions evaluated by the network\n    simulations: int\n    depth: int\n    seconds: float\n    pv: list\n    scores: list                 # (move, network win chance, visits) at the root, best first\n\n\nclass MCTS:\n    def __init__(self, model, c_puct=1.5, prior_temp=0.03, batch_leaves=4):\n        self.model, self.c_puct, self.prior_temp, self.batch_leaves = model, c_puct, prior_temp, batch_leaves\n        self.root, self.root_board = None, None\n\n    # ---------- tree building ----------\n    def _make_node(self, board, moves, q):\n        q = q.astype(np.float64).copy()\n        exact = np.full(len(moves), np.nan)\n        for i, mv in enumerate(moves):          # exact results one ply ahead (mate-in-1, stalemate, ...)\n            board.push(mv)\n            v = terminal_value(board)\n            board.pop()\n            if v is not None:\n                exact[i] = 1.0 - v              # value for the side that played mv\n        z = (np.where(np.isnan(exact), q, exact) - q.max()) / self.prior_temp\n        prior = np.exp(z - z.max()); prior /= prior.sum()\n        return Node(moves, q, prior, np.zeros(len(moves)), np.zeros(len(moves)), exact)\n\n    def set_position(self, board):\n        """Start from `board`, reusing the subtree from the previous search when the game continued."""\n        node = None\n        if self.root is not None and self.root_board is not None:\n            old, new = self.root_board.move_stack, board.move_stack\n            if len(new) >= len(old) and new[:len(old)] == old and board.root() == self.root_board.root():\n                node = self.root\n                for mv in new[len(old):]:\n                    node = node.children.get(mv) if node is not None else None\n        self.root_board = board.copy()\n        if node is None:\n            moves, q = self.model.evaluate([board])[0]\n            node = self._make_node(board, moves, q)\n        self.root = node\n        return node\n\n    # ---------- one batch of simulations ----------\n    def _select(self, node):\n        q = node.q()\n        u = self.c_puct * node.prior * math.sqrt(node.visits + 1) / (1.0 + node.N)\n        return int(np.argmax(q + u))\n\n    def _simulate_batch(self, max_leaves=None):\n        """Walk up to `batch_leaves` paths with virtual loss, evaluate their leaves in one network call, back up."""\n        pending, boards = [], []\n        for _ in range(min(self.batch_leaves, max_leaves or self.batch_leaves)):\n            board, node, path = self.root_board.copy(), self.root, []\n            while True:\n                a = self._select(node)\n                path.append((node, a))\n                node.N[a] += 1; node.visits += 1            # virtual loss: a visit with value 0 until backed up\n                if not np.isnan(node.exact[a]):             # game over after this move: exact value\n                    pending.append((path, None, node.exact[a], len(path)))\n                    break\n                mv = node.moves[a]\n                board.push(mv)\n                child = node.children.get(mv)\n                if child is None:\n                    tv = terminal_value(board)              # e.g. threefold repetition reached in the tree\n                    if tv is not None:\n                        pending.append((path, None, 1.0 - tv, len(path)))\n                    else:\n                        pending.append((path, board, None, len(path)))\n                        boards.append(board)\n                    break\n                node = child\n        unique = {}\n        for b in boards:                                    # several paths can reach the same new position\n            unique.setdefault(b._transposition_key(), b)\n        evals = self.model.evaluate(list(unique.values())) if unique else []\n        expanded = {key: self._make_node(b, *ev) for (key, b), ev in zip(unique.items(), evals)}\n        depth = 0\n        for path, board, value, d in pending:\n            depth = max(depth, d)\n            if board is not None:\n                leaf = expanded[board._transposition_key()]\n                parent, a = path[-1]\n                parent.children.setdefault(parent.moves[a], leaf)\n                value = 1.0 - float(np.max(leaf.q()))       # leaf value for the side that moved into it\n            for parent, a in reversed(path):                # alternate perspective on the way up\n                parent.W[a] += value\n                value = 1.0 - value\n        return len(unique), len(pending), depth\n\n    # ---------- public API ----------\n    def search(self, board, nodes=None, seconds=None, stop_event=None, info=None, info_every=1.0):\n        """Search `board` until `nodes` network evaluations or `seconds` elapsed (whichever first).\n\n        nodes=0 -> no search: play the network\'s best move.\n        """\n        t0 = time.time()\n        root = self.set_position(board)\n        if not root.moves:\n            raise ValueError("no legal moves")\n        n_eval = n_sims = depth = 0\n        last_info = t0\n        batch_s = 0.0                                       # duration of the last batch, to avoid overrunning the clock\n        if nodes != 0 and len(root.moves) > 1:\n            while True:\n                if stop_event is not None and stop_event.is_set():\n                    break\n                if nodes is not None and n_eval >= nodes:\n                    break\n                if seconds is not None and time.time() - t0 + 0.8 * batch_s >= seconds:\n                    break\n                tb = time.time()\n                e, s, d = self._simulate_batch(None if nodes is None else nodes - n_eval)\n                batch_s = time.time() - tb\n                n_eval += e; n_sims += s; depth = max(depth, d)\n                if np.nanmax(np.where(np.isnan(root.exact), -1, root.exact)) == WIN:\n                    break                                   # forced mate in one: nothing to think about\n                if info is not None and time.time() - last_info >= info_every:\n                    info(self._result(t0, n_eval, n_sims, depth)); last_info = time.time()\n        res = self._result(t0, n_eval, n_sims, depth)\n        if info is not None:\n            info(res)\n        return res\n\n    def _result(self, t0, n_eval, n_sims, depth):\n        root = self.root\n        q = root.q()\n        if root.N.sum() > 0:\n            a = int(np.lexsort((q, root.N))[-1])           # most visits, ties broken by value\n        else:\n            a = int(np.argmax(q))\n        pv, node, b = [root.moves[a]], root.children.get(root.moves[a]), None\n        while node is not None and node.N.sum() > 0:\n            i = int(np.argmax(node.N)); pv.append(node.moves[i]); node = node.children.get(node.moves[i])\n        order = np.lexsort((q, root.N))[::-1]\n        scores = [(root.moves[i], float(root.q_net[i]), int(root.N[i])) for i in order]\n        return SearchResult(root.moves[a], float(q[a]), n_eval, n_sims, depth, time.time() - t0, pv, scores)\n\n\nclass SearchThread:\n    """Runs a search in the background so a UCI `stop` can interrupt it."""\n\n    def __init__(self, mcts):\n        self.mcts, self.stop_event, self.thread, self.result = mcts, threading.Event(), None, None\n\n    def start(self, board, done, **kw):\n        self.stop_event.clear()\n\n        def run():\n            self.result = self.mcts.search(board, stop_event=self.stop_event, **kw)\n            done(self.result)\n\n        self.thread = threading.Thread(target=run, daemon=True)\n        self.thread.start()\n\n    def stop(self):\n        self.stop_event.set()\n        if self.thread is not None:\n            self.thread.join()\n', 'match.py': '"""LayaChess vs Stockfish (strength-limited) match with an Elo estimate.\n\n    python -m laya_chess.match --games 24 --sf-elo 1320 --laya-nodes 0          # network only\n    python -m laya_chess.match --games 24 --sf-elo 1320 --laya-seconds 10       # with search\n\nEach opening is played twice with colours swapped. Games are appended to a PGN as they finish.\n"""\nimport argparse\nimport datetime\nimport math\nimport shutil\nimport time\n\nimport chess\nimport chess.engine\nimport chess.pgn\n\nfrom .model import DEFAULT_CHECKPOINT, LayaChessModel\nfrom .search import MCTS, winprob_to_cp\n\n# short, mainstream openings (UCI), so games start from normal, varied positions\nOPENINGS = [\n    ("Italian", "e2e4 e7e5 g1f3 b8c6 f1c4"),\n    ("Sicilian", "e2e4 c7c5 g1f3 d7d6"),\n    ("French", "e2e4 e7e6 d2d4 d7d5"),\n    ("Caro-Kann", "e2e4 c7c6 d2d4 d7d5"),\n    ("Queen\'s Gambit Declined", "d2d4 d7d5 c2c4 e7e6"),\n    ("King\'s Indian", "d2d4 g8f6 c2c4 g7g6"),\n    ("Nimzo/Queen\'s Indian setup", "d2d4 g8f6 c2c4 e7e6"),\n    ("English", "c2c4 e7e5 b1c3 g8f6"),\n    ("Reti", "g1f3 d7d5 g2g3 g8f6"),\n    ("Two Knights", "e2e4 e7e5 f1c4 g8f6"),\n    ("London", "d2d4 d7d5 c1f4 g8f6"),\n    ("Pirc", "e2e4 d7d6 d2d4 g8f6"),\n]\n\n\ndef elo_from_score(score):\n    score = min(max(score, 1e-3), 1 - 1e-3)\n    return -400 * math.log10(1 / score - 1)\n\n\ndef summary(results, sf_elo):\n    """results: list of 1 / 0.5 / 0 from Laya\'s point of view."""\n    n = len(results)\n    w, d, l = results.count(1.0), results.count(0.5), results.count(0.0)\n    s = sum(results) / n\n    var = sum((x - s) ** 2 for x in results) / max(1, n - 1)\n    se = math.sqrt(var / n) if n > 1 else 0.5\n    lo, hi = max(1e-3, s - 1.96 * se), min(1 - 1e-3, s + 1.96 * se)\n    diff = elo_from_score(s)\n    return (f"{n} games  +{w} ={d} -{l}  score {s:.1%}  Elo diff {diff:+.0f} "\n            f"[{elo_from_score(lo):+.0f}, {elo_from_score(hi):+.0f}]  -> performance ~{sf_elo + diff:.0f} "\n            f"(Stockfish UCI_Elo scale)")\n\n\nclass LayaPlayer:\n    def __init__(self, model, nodes, seconds, batch_leaves):\n        self.mcts = MCTS(model, batch_leaves=batch_leaves)\n        self.nodes, self.seconds = nodes, seconds\n\n    def new_game(self):\n        self.mcts.root = self.mcts.root_board = None\n\n    def play(self, board):\n        r = self.mcts.search(board, nodes=self.nodes, seconds=self.seconds)\n        return r.move, f"laya {r.value:.1%} cp {winprob_to_cp(r.value)} n {r.nodes} {r.seconds:.1f}s"\n\n\ndef play_game(laya, sf, sf_limit, opening, laya_white, max_plies):\n    board = chess.Board()\n    for uci in opening[1].split():\n        board.push_uci(uci)\n    laya.new_game()\n    comments = {}\n    while not board.is_game_over(claim_draw=True) and board.ply() < max_plies:\n        if (board.turn == chess.WHITE) == laya_white:\n            mv, note = laya.play(board)\n            comments[board.ply()] = note\n        else:\n            mv = sf.play(board, sf_limit).move\n        board.push(mv)\n    outcome = board.outcome(claim_draw=True)\n    result = outcome.result() if outcome else "1/2-1/2"          # move cap -> adjudicated draw\n    return board, result, comments\n\n\ndef to_pgn(board, result, comments, opening, white, black, event):\n    game = chess.pgn.Game()\n    game.headers.update({"Event": event, "Site": "local", "Date": datetime.date.today().strftime("%Y.%m.%d"),\n                         "White": white, "Black": black, "Result": result, "Opening": opening[0]})\n    node = game\n    replay = chess.Board()\n    for mv in board.move_stack:\n        ply = replay.ply()\n        node = node.add_variation(mv)\n        if ply in comments:\n            node.comment = comments[ply]\n        replay.push(mv)\n    return game\n\n\ndef main():\n    p = argparse.ArgumentParser(description="LayaChess vs Stockfish match")\n    p.add_argument("--games", type=int, default=24, help="even number; each opening twice with colours swapped")\n    p.add_argument("--sf-elo", type=int, default=1320, help="Stockfish UCI_Elo (min 1320)")\n    p.add_argument("--sf-time", type=float, default=0.5, help="Stockfish seconds per move")\n    p.add_argument("--stockfish", default=shutil.which("stockfish") or "stockfish")\n    p.add_argument("--checkpoint", default=DEFAULT_CHECKPOINT)\n    p.add_argument("--revision", default=None)\n    p.add_argument("--device", default="auto")\n    p.add_argument("--laya-nodes", type=int, default=None, help="evaluations per move (0 = network only)")\n    p.add_argument("--laya-seconds", type=float, default=None, help="thinking time per move (search)")\n    p.add_argument("--batch-leaves", type=int, default=1)\n    p.add_argument("--max-plies", type=int, default=300)\n    p.add_argument("--pgn", default=None, help="output PGN (default: matches/<settings>.pgn)")\n    a = p.parse_args()\n    if a.laya_nodes is None and a.laya_seconds is None:\n        a.laya_nodes = 0\n\n    mode = (f"nodes{a.laya_nodes}" if a.laya_nodes is not None and a.laya_seconds is None else f"{a.laya_seconds:g}s")\n    laya_name = f"LayaChess ({mode})"\n    sf_name = f"Stockfish (UCI_Elo {a.sf_elo})"\n    pgn_path = a.pgn or f"matches/laya-{mode}_vs_sf{a.sf_elo}.pgn"\n    import os\n    os.makedirs(os.path.dirname(pgn_path) or ".", exist_ok=True)\n\n    model = LayaChessModel(a.checkpoint, revision=a.revision, device=a.device)\n    laya = LayaPlayer(model, a.laya_nodes, a.laya_seconds, a.batch_leaves)\n    sf = chess.engine.SimpleEngine.popen_uci(a.stockfish)\n    sf.configure({"UCI_LimitStrength": True, "UCI_Elo": a.sf_elo, "Threads": 1, "Hash": 16})\n    sf_limit = chess.engine.Limit(time=a.sf_time)\n\n    results = []\n    t0 = time.time()\n    try:\n        for g in range(a.games):\n            opening = OPENINGS[(g // 2) % len(OPENINGS)]\n            laya_white = g % 2 == 0\n            tg = time.time()\n            board, result, comments = play_game(laya, sf, sf_limit, opening, laya_white, a.max_plies)\n            score = {"1-0": 1.0, "0-1": 0.0}.get(result, 0.5)\n            results.append(score if laya_white else 1.0 - score)\n            white, black = (laya_name, sf_name) if laya_white else (sf_name, laya_name)\n            with open(pgn_path, "a") as f:\n                print(to_pgn(board, result, comments, opening, white, black, f"{laya_name} vs {sf_name}"), file=f, end="\\n\\n")\n            print(f"game {g + 1}/{a.games} {opening[0]:26s} Laya {\'white\' if laya_white else \'black\'}: {result:7s} "\n                  f"{board.ply()} plies {time.time() - tg:.0f}s | {summary(results, a.sf_elo)}", flush=True)\n    finally:\n        sf.quit()\n    print(f"\\nFINAL {laya_name} vs {sf_name}: {summary(results, a.sf_elo)}")\n    print(f"PGN: {pgn_path} | total {(time.time() - t0) / 60:.0f} min")\n\n\nif __name__ == "__main__":\n    main()\n', '__init__.py': '"""LayaChess engine (embedded copy)."""\n'}
for name, src in FILES.items():
    open(f'/kaggle/working/engine/laya_chess/{name}', 'w').write(src)
print('wrote', sorted(FILES))

## Worker: one per GPU, plays its share of the games

In [ ]:
WORKER = r"""
import json, sys, time, chess, chess.engine
sys.path.insert(0, "/kaggle/working/engine")
from laya_chess.model import LayaChessModel
from laya_chess.match import LayaPlayer, OPENINGS, play_game, to_pgn

cfg = json.loads(sys.argv[1]); wid = cfg["worker"]
model = LayaChessModel(cfg["checkpoint"], revision=cfg["revision"], device="cuda", batch_size=192)
laya = LayaPlayer(model, cfg["nodes"], None, batch_leaves=4)
sf = chess.engine.SimpleEngine.popen_uci("stockfish")
deadline = time.time() + cfg["budget_s"]
out = open(f"/kaggle/working/matches/results_w{wid}.jsonl", "a")
for level, g in cfg["tasks"]:
    if time.time() > deadline:
        print(f"[w{wid}] time budget reached", flush=True); break
    sf.configure({"UCI_LimitStrength": True, "UCI_Elo": level, "Threads": 1, "Hash": 16})
    opening = OPENINGS[(g // 2) % len(OPENINGS)]; laya_white = g % 2 == 0
    t0, n0 = time.time(), model.n_positions
    board, result, comments = play_game(laya, sf, chess.engine.Limit(time=cfg["sf_seconds"]), opening, laya_white, cfg["max_plies"])
    s = {"1-0": 1.0, "0-1": 0.0}.get(result, 0.5); score = s if laya_white else 1 - s
    dt = time.time() - t0
    rec = {"level": level, "game": g, "opening": opening[0], "laya_white": laya_white, "result": result, "score": score,
           "plies": board.ply(), "seconds": round(dt, 1), "positions_per_s": round((model.n_positions - n0) / dt, 2)}
    out.write(json.dumps(rec) + "\n"); out.flush()
    names = ("LayaChess+search", f"Stockfish {level}") if laya_white else (f"Stockfish {level}", "LayaChess+search")
    with open(f"/kaggle/working/matches/laya_vs_sf{level}_w{wid}.pgn", "a") as f:
        print(to_pgn(board, result, comments, opening, *names, f"LayaChess (nodes {cfg['nodes']}) vs Stockfish UCI_Elo {level}"), file=f, end="\n\n")
    print(f"[w{wid}] SF{level} game {g:2d} {opening[0]:26s} Laya {'W' if laya_white else 'B'} {result:7s} "
          f"{board.ply()//2} moves {dt/60:.1f} min ({rec['positions_per_s']} pos/s)", flush=True)
sf.quit()
"""
open("/kaggle/working/engine/worker.py", "w").write(WORKER)
print("worker written")

## Run both GPUs and show the scoreboard while it plays

In [ ]:
import json, glob, math, os, subprocess, sys, time
os.makedirs("/kaggle/working/matches", exist_ok=True)

# interleave levels so every level gets games even if the time budget runs out
tasks = [(lvl, g) for g in range(GAMES_PER_LEVEL) for lvl in LEVELS]
n_gpu = 2
procs = []
for w in range(n_gpu):
    cfg = {"worker": w, "tasks": tasks[w::n_gpu], "nodes": LAYA_NODES, "sf_seconds": SF_SECONDS,
           "budget_s": TIME_BUDGET_H * 3600, "checkpoint": CHECKPOINT, "revision": REVISION, "max_plies": MAX_PLIES}
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(w), TOKENIZERS_PARALLELISM="false")
    log = open(f"/kaggle/working/matches/worker{w}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "/kaggle/working/engine/worker.py", json.dumps(cfg)], env=env, stdout=log, stderr=subprocess.STDOUT))
print(f"{len(tasks)} games queued on {n_gpu} GPUs")

def results():
    out = []
    for p in glob.glob("/kaggle/working/matches/results_w*.jsonl"):
        out += [json.loads(l) for l in open(p) if l.strip()]
    return out

def elo(s): s = min(max(s, 1e-3), 1 - 1e-3); return -400 * math.log10(1 / s - 1)

def scoreboard(rs):
    lines = []
    for lvl in LEVELS:
        r = [x["score"] for x in rs if x["level"] == lvl]
        if not r: continue
        n, s = len(r), sum(r) / len(r)
        lines.append(f"  SF {lvl}: {n:2d} games +{r.count(1.0)} ={r.count(0.5)} -{r.count(0.0)}  score {s:5.1%}  -> perf ~{lvl + elo(s):.0f}")
    return "\n".join(lines)

def backup(note):
    # copy every finished game to the HF repo, so a killed session loses at most the games in progress
    if not UPLOAD_TO_HF:
        return
    try:
        from huggingface_hub import HfApi
        HfApi(token=os.environ["HF_TOKEN"]).upload_folder(
            folder_path="/kaggle/working/matches", repo_id=CHECKPOINT, path_in_repo="matches",
            allow_patterns=["*.pgn", "*.json", "*.jsonl", "*.png", "*.log"], commit_message=note)
        print(f"[backup] {note} -> https://huggingface.co/{CHECKPOINT}/tree/main/matches", flush=True)
    except Exception as e:
        print("[backup] failed (will retry next time):", e, flush=True)

t0, seen, last_backup = time.time(), 0, time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    rs = results()
    if len(rs) != seen:
        seen = len(rs)
        pps = [x["positions_per_s"] for x in rs]
        print(f"--- {(time.time() - t0) / 3600:.2f} h | {len(rs)}/{len(tasks)} games | ~{sum(pps)/len(pps):.1f} positions/s per GPU ---")
        print(scoreboard(rs), flush=True)
    if time.time() - last_backup > BACKUP_EVERY_MIN * 60 and rs:
        backup(f"matches (in progress): {len(rs)} games"); last_backup = time.time()
backup(f"matches (all workers done): {len(results())} games")
for w in range(n_gpu):
    print(f"worker {w} exit code {procs[w].returncode}; last log lines:")
    print("".join(open(f"/kaggle/working/matches/worker{w}.log").readlines()[-3:]))

## Final results: score per level + overall performance rating

In [ ]:
rs = results()
print(f"{len(rs)} games played\n" + scoreboard(rs))

# overall performance rating: the rating R at which expected score vs all opponents equals the actual score
def expected(R): return sum(1 / (1 + 10 ** ((x["level"] - R) / 400)) for x in rs)
actual = sum(x["score"] for x in rs)
lo, hi = 0.0, 4000.0
for _ in range(60):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if expected(mid) < actual else (lo, mid)
perf = (lo + hi) / 2
print(f"\nOverall performance rating: ~{perf:.0f} on Stockfish's UCI_Elo scale "
      f"({actual:g}/{len(rs)} points, Laya searching {LAYA_NODES} positions per move)")

summary = {"levels": {}, "performance": round(perf), "games": len(rs), "laya_nodes": LAYA_NODES,
           "sf_seconds": SF_SECONDS, "checkpoint": CHECKPOINT, "revision": REVISION}
for lvl in LEVELS:
    r = [x["score"] for x in rs if x["level"] == lvl]
    if r:
        summary["levels"][lvl] = {"games": len(r), "wins": r.count(1.0), "draws": r.count(0.5), "losses": r.count(0.0),
                                  "score": round(sum(r) / len(r), 3)}
json.dump(summary, open("/kaggle/working/matches/summary.json", "w"), indent=1)

import matplotlib.pyplot as plt
lv = [l for l in LEVELS if l in summary["levels"]]
sc = [summary["levels"][l]["score"] * 100 for l in lv]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(l) for l in lv], sc, color="#769656")
ax.axhline(50, color="#888", lw=1, ls="--")
for i, v in enumerate(sc):
    ax.text(i, v + 1.5, f"{v:.0f}%", ha="center")
ax.set_ylim(0, 105); ax.set_xlabel("Stockfish UCI_Elo"); ax.set_ylabel("LayaChess score (%)")
ax.set_title(f"LayaChess + search ({LAYA_NODES} positions/move) vs Stockfish — perf ~{perf:.0f}")
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig("/kaggle/working/matches/score_by_level.png", dpi=160); plt.show()

In [ ]:
if UPLOAD_TO_HF:
    try:
        from huggingface_hub import HfApi
        HfApi(token=os.environ["HF_TOKEN"]).upload_folder(
            folder_path="/kaggle/working/matches", repo_id=CHECKPOINT, path_in_repo="matches",
            allow_patterns=["*.pgn", "*.json", "*.jsonl", "*.png"],
            commit_message=f"matches: {len(rs)} games vs Stockfish, perf ~{perf:.0f}")
        print(f"uploaded to https://huggingface.co/{CHECKPOINT}/tree/main/matches")
    except Exception as e:
        print("HF upload failed (results are still in /kaggle/working/matches):", e)